# CoT Distillation v2 — Kaggle Pipeline

Reproduces Magister et al. (ACL 2023) at small scale on FLAN-T5-base, plus
ReCEval (Prasad et al., EMNLP 2023) as a second evaluation axis.

**This notebook is the v2 successor to `cot-gpt.ipynb`.** v1 ran end-to-end
but produced post-distillation accuracy *below* the zero-shot baseline
(see `doc/Current Notebook.md` in the repo). v2 fixes the recipe (lr 3e-4 →
5e-5, weight_decay 0 → 0.01, epochs 3 → 8 with early stopping, max_seq
256 → 512) and the decoding (greedy → beam=4 + no_repeat_ngram=4 +
repetition_penalty=1.15). It also adds two conditions (Direct FT and a
calculator-corrected filter, Set C).

**Authoritative spec:** `AGENT.md` in the repo.

## How to use this notebook

1. Run the **Setup** cells once per session to clone/pull and install deps.
2. Run the **Config** cell to confirm the v2 recipe; tweak inline if you
   want to A/B a hyperparameter (it patches `config/config.yaml` in place).
3. Run stages in order. Each stage cell prints a `python -m src.status`
   summary at the end so you can see where the project stands without
   scrolling.
4. The four real Stage-3 runs are split into separate cells so you can
   run them one at a time across Kaggle sessions, resuming with `--resume`.
5. **Persist outputs:** Kaggle's `/kaggle/working` survives only the current
   session. The final cell zips and uploads `outputs/checkpoints/`,
   `outputs/generations/`, and `outputs/runs/` to a Kaggle Dataset so you
   can pick up across sessions. Configure your Kaggle username + dataset
   slug in the Config cell.

## 0 · Setup

In [ ]:
# 0.1 — Clone OR pull the repo. Idempotent: run every session.
import os, subprocess, pathlib

REPO_URL  = "https://github.com/rihembenabdallah18/COT_lab.git"
BRANCH    = "dev"      # switch to 'main' once v2 is merged
REPO_NAME = "COT_lab"
WORKDIR   = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle").exists() else pathlib.Path.cwd()
REPO_DIR  = WORKDIR / REPO_NAME

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("repo:", REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-5"], check=True)

In [ ]:
# 0.2 — Install dependencies.
# torch is preinstalled on Kaggle; we install the rest from requirements.txt.
# `peft` if present can shadow imports — uninstall to be safe.
!pip install -q -r requirements.txt
!pip uninstall -y -q peft || true
!python -m spacy download en_core_web_sm -q

In [ ]:
# 0.3 — GPU check. Should print True + Tesla T4 (or P100 on Kaggle premium).
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("mem (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1 · Config — review and (optionally) override

The single source of truth is `config/config.yaml`. The cell below loads it,
shows the v2 recipe, and lets you patch any field for an ablation without
editing the file by hand. Set `OVERRIDES = {}` to keep the committed v2
values (recommended for the first run).

In [ ]:
import yaml, pathlib, json

CFG_PATH = pathlib.Path("config/config.yaml")
cfg = yaml.safe_load(CFG_PATH.read_text())

# ---- Edit here for ablations ----------------------------------------------
OVERRIDES = {
    # "learning_rate": 3.0e-5,
    # "num_epochs": 6,
    # "max_input_length": 384,
    # "max_target_length": 384,
    # "inference_num_beams": 5,
}
# ---------------------------------------------------------------------------

# Where to back up `outputs/` between sessions.
KAGGLE_USERNAME       = "YOUR_KAGGLE_USERNAME"     # <-- set this
BACKUP_DATASET_SLUG   = "cot-lab-v2-outputs"        # <-- and this

for k, v in OVERRIDES.items():
    if k not in cfg:
        raise KeyError(f"Unknown config key: {k}")
    cfg[k] = v

if OVERRIDES:
    CFG_PATH.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print("applied overrides:", OVERRIDES)

print(json.dumps({k: cfg[k] for k in [
    "model_name", "learning_rate", "weight_decay", "warmup_ratio",
    "num_epochs", "early_stopping_patience", "max_input_length",
    "max_target_length", "batch_size", "gradient_accumulation_steps",
    "fp16", "inference_num_beams", "inference_no_repeat_ngram_size",
    "inference_repetition_penalty", "inference_max_new_tokens", "seed"
]}, indent=2))

## 2 · Project status (run any time)

Reads `outputs/runs/*.json` and prints one row per stage run. Use this
between cells to see what's done.

In [ ]:
!python -m src.status

## 3 · Stage 1 — Download GSM8K + Ho et al. teacher CoTs

Idempotent: skips files already on disk. Reports the Ho et al. JSON schema
and prints 5 example train rows + their teacher CoT.

In [ ]:
!bash scripts/01_download.sh

## 4 · Stage 2 — Build the four training sets

Produces `direct_ft.jsonl`, `set_a_nofilter.jsonl`, `set_b_magister.jsonl`,
`set_c_calculator.jsonl`. Set C is the calculator-corrected, *process-aware*
filter — see `AGENT.md §4`. Writes a Stage-2 run-card.

In [ ]:
!bash scripts/02_filter.sh
!python -m src.status

### 4.1 — Unit tests

27 cases covering the answer parser and the calculator. Run before any
training to catch regressions if you bumped a library version.

In [ ]:
!python -m pytest tests/ -q

## 5 · Stage 3 — Fine-tune the four students

Run the cells **in order**. Each is independent and resumable
(`--resume` picks up from the latest checkpoint after a session timeout).

Per AGENT.md, run cheapest-first so the recipe is validated before the
longest run:

1. **Smoke** (200 ex, 1 epoch) — verifies the loop end-to-end
2. **Direct FT** — Ho et al. 4.93% reference; cheapest real run
3. **Set B** (Magister filter, 3,389)
4. **Set C** (calculator-corrected, 2,635)
5. **Set A** (no filter, 7,473) — longest

After Direct FT finishes, **stop and inspect the run-card.** If `best_eval_loss`
is well above ~1.0 the recipe is still wrong — escalate before the longer runs.

### 5.1 — Smoke (200 ex, 1 epoch, ~10 min on T4)

In [ ]:
!bash scripts/03_train_smoke.sh

### 5.2 — Direct FT (Q→A only, no CoT)

Recipe-validation gate. After this finishes, check the run-card.

In [ ]:
!bash scripts/03_train_direct_ft.sh
!python -m src.status

### 5.3 — Set B (Magister filter)

In [ ]:
!bash scripts/03_train_set_b.sh
!python -m src.status

### 5.4 — Set C (calculator-corrected, process-aware)

In [ ]:
!bash scripts/03_train_set_c.sh
!python -m src.status

### 5.5 — Set A (no filter, longest run)

In [ ]:
!bash scripts/03_train_set_a.sh
!python -m src.status

## 6 · Stage 4 — Inference on the GSM8K test set

Runs all five conditions with v2 decoding (beam=4, no_repeat_ngram=4,
repetition_penalty=1.15). Each writes a JSONL and a run-card. Resumable.

In [ ]:
!bash scripts/04_inference.sh
!python -m src.status

## 7 · Stage 5 — Evaluation (placeholder cells)

These will be implemented in upcoming commits. Re-pull the repo (cell 0.1)
before running.

In [ ]:
# 7.1 — Accuracy + accuracy-with-calculator (Stage 5a)
# !bash scripts/05a_accuracy.sh
# !cat outputs/eval_results/accuracy.csv

In [ ]:
# 7.2 — ReCEval (Stage 5b)
# !bash scripts/05b_receval.sh
# !cat outputs/eval_results/receval_summary.csv

## 8 · Persist outputs across sessions

Kaggle wipes `/kaggle/working` between sessions for free notebooks unless
you save it to a Kaggle Dataset. The cell below packages
`outputs/checkpoints/`, `outputs/generations/`, `outputs/runs/`, and
`data/processed/` and uploads (or updates) the dataset configured at the
top of this notebook.

Requires `KAGGLE_USERNAME` + `KAGGLE_KEY` in Kaggle Secrets, or
`~/.kaggle/kaggle.json`.

In [ ]:
import json, os, pathlib, shutil, subprocess

STAGE_DIR = pathlib.Path("/kaggle/working/_dataset_payload")
STAGE_DIR.mkdir(parents=True, exist_ok=True)
for src in ["outputs/checkpoints", "outputs/generations", "outputs/runs",
            "outputs/eval_results", "data/processed"]:
    src_path = pathlib.Path(src)
    if src_path.exists():
        dst = STAGE_DIR / src_path.name
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src_path, dst)
        print("staged:", src, "->", dst)

# Write a dataset-metadata.json so kaggle datasets create/update can identify it.
meta = {
    "title":      "COT_lab v2 outputs",
    "id":         f"{KAGGLE_USERNAME}/{BACKUP_DATASET_SLUG}",
    "licenses":   [{"name": "CC0-1.0"}],
}
(STAGE_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

if KAGGLE_USERNAME == "YOUR_KAGGLE_USERNAME":
    print("Set KAGGLE_USERNAME / BACKUP_DATASET_SLUG in the Config cell first.")
else:
    # First time:  kaggle datasets create -p <dir> -r zip
    # Updates:     kaggle datasets version -p <dir> -m '...' -r zip
    creator = subprocess.run(["kaggle", "datasets", "version", "-p", str(STAGE_DIR),
                              "-m", "v2 outputs update", "-r", "zip"],
                             capture_output=True, text=True)
    if creator.returncode != 0 and "Not Found" in (creator.stderr + creator.stdout):
        creator = subprocess.run(["kaggle", "datasets", "create", "-p", str(STAGE_DIR),
                                  "-r", "zip"], capture_output=True, text=True)
    print(creator.stdout); print(creator.stderr)